In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')


In [ ]:
data_path = '/kaggle/input/datasets/lohithr2004/mimic42/mimic_3drug_expanded (1).csv'

df = pd.read_csv(data_path)

print(df.head())

In [ ]:
print(df.shape)

In [ ]:
df_filtered = df.copy()
df_filtered = df_filtered[df_filtered['anchor_age'] >= 18]

In [ ]:
# AGE DISTRIBUTION OF PATIENTS

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

plt.figure(figsize=(9,5), dpi=150)

sns.histplot(
    df_filtered['anchor_age'],
    bins=20,
    kde=True,
    color='skyblue',
    edgecolor='black',
    line_kws={
        'color': 'darkblue',
        'linewidth': 2
    }
)


plt.title(
    'Age Distribution of Patients',
    fontsize=18,
    fontweight='bold',
    pad=12
)


plt.xlabel(
    'Age',
    fontsize=13,
    fontweight='bold'
)

plt.ylabel(
    'Number of Patients',
    fontsize=13,
    fontweight='bold'
)


plt.xticks(fontsize=11)
plt.yticks(fontsize=11)


plt.grid(
    alpha=0.3,
    linestyle='--'
)


sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
# GENDER DISTRIBUTION OF PATIENTS

import matplotlib.pyplot as plt


plt.figure(figsize=(6,5))


gender_counts = df_filtered['gender_encoded'].value_counts()


plt.pie(
    gender_counts,
    labels=['Male', 'Female'],
    autopct='%1.1f%%'
)


plt.title(
    'Gender Distribution',
    fontsize=16,
    fontweight='bold'
)


plt.tight_layout()


plt.show()

In [ ]:
# PHARMACOKINETIC CALCULATIONS

print("Calculating Creatinine Clearance...")


is_female = df['gender_encoded'] == 0

# Cockcroft-Gault Equation
df['crcl'] = (
    (
        (140 - df['anchor_age']) *
        df['weight_kg']
    ) /
    (72 * df['creatinine'])
)


df.loc[is_female, 'crcl'] *= 0.85

print("Creatinine Clearance calculation completed!")

In [ ]:
# VANCOMYCIN PPK MODEL

print("Calculating Vancomycin PPK estimates...")

vanco_clearance = 0.65 * df['crcl']

vanco_dose = 1000

df['ppk_vanco_estimate'] = (
    vanco_dose /
    (vanco_clearance + 10)
)

print("Vancomycin PPK completed!")

In [ ]:
# GENTAMICIN PPK MODEL

print("Calculating Gentamicin PPK estimates...")

genta_clearance = 0.8 * df['crcl']

genta_dose = 400

df['ppk_genta_estimate'] = (
    genta_dose /
    (genta_clearance + 5)
)

print("Gentamicin PPK completed!")

In [ ]:
# PHENYTOIN PPK MODEL

print("Calculating Phenytoin PPK estimates...")

dose_pheny = 300
km_pheny = 4.0

# Standard healthy liver metabolism
vmax_standard = (
    7.0 * df['weight_kg']
)

# Adjusting for albumin + liver health
adjusted_vmax = (
    vmax_standard *
    (df['albumin'] / 4.0) *
    df['liver_health']
)

df['ppk_pheny_estimate'] = (
    (km_pheny * dose_pheny) /
    np.clip((adjusted_vmax - dose_pheny), 40, None)
)

df['ppk_pheny_estimate'] = np.clip(
    df['ppk_pheny_estimate'],
    5,
    25
)

print("Phenytoin PPK completed!")

In [ ]:
# CALCULATE RESIDUAL ERRORS

df['error_vanco'] = (
    df['actual_vanco_conc'] -
    df['ppk_vanco_estimate']
)

df['error_genta'] = (
    df['actual_genta_conc'] -
    df['ppk_genta_estimate']
)

df['error_pheny'] = (
    df['actual_pheny_conc'] -
    df['ppk_pheny_estimate']
)

In [ ]:

df = df[
    
    # VANCOMYCIN
    (df['actual_vanco_conc'] >= 5) &
    (df['actual_vanco_conc'] <= 35) &
    
    # GENTAMICIN
    (df['actual_genta_conc'] >= 1) &
    (df['actual_genta_conc'] <= 12) &
    
    # PHENYTOIN
    (df['actual_pheny_conc'] >= 5) &
    (df['actual_pheny_conc'] <= 25)
]



In [ ]:
df_filtered = df.copy()

df_filtered = df_filtered[

    (df_filtered['anchor_age'] >= 18) &
    (df_filtered['weight_kg'] > 20) &
    (df_filtered['mean_bp'] > 30) &
    (df_filtered['heart_rate'] > 20) &
    (df_filtered['heart_rate'] <= 300) &
    (df_filtered['temperature_c'] >= 30.0) &
    (df_filtered['temperature_c'] <= 45.0) &
    (df_filtered['wbc_count'] > 0) &
    (df_filtered['creatinine'] > 0.1)

]

print(df_filtered.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=150)
fig.suptitle('Target Variable Distributions: Actual Concentrations & Residual Errors', 
             fontsize=20, fontweight='bold', y=1.02)


# ROW 1: Actual Drug Concentrations

# Vancomycin
sns.histplot(df_filtered['actual_vanco_conc'], bins=25, kde=True, 
             ax=axes[0, 0], color='skyblue', edgecolor='black')
axes[0, 0].set_title('Actual Vancomycin Conc.', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Concentration (ug/mL)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)

# Gentamicin
sns.histplot(df_filtered['actual_genta_conc'], bins=25, kde=True, 
             ax=axes[0, 1], color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Actual Gentamicin Conc.', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Concentration (ug/mL)', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)

# Phenytoin
sns.histplot(df_filtered['actual_pheny_conc'], bins=25, kde=True, 
             ax=axes[0, 2], color='salmon', edgecolor='black')
axes[0, 2].set_title('Actual Phenytoin Conc.', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('Concentration (ug/mL)', fontsize=12)
axes[0, 2].set_ylabel('Frequency', fontsize=12)



# Vancomycin Error
sns.histplot(df_filtered['error_vanco'], bins=25, kde=True, 
             ax=axes[1, 0], color='dodgerblue', edgecolor='black')
axes[1, 0].set_title('Vancomycin Residual Error', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Error (Actual - PPK)', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)

# Gentamicin Error
sns.histplot(df_filtered['error_genta'], bins=25, kde=True, 
             ax=axes[1, 1], color='forestgreen', edgecolor='black')
axes[1, 1].set_title('Gentamicin Residual Error', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Error (Actual - PPK)', fontsize=12)
axes[1, 1].set_ylabel('Frequency', fontsize=12)

# Phenytoin Error
sns.histplot(df_filtered['error_pheny'], bins=25, kde=True, 
             ax=axes[1, 2], color='firebrick', edgecolor='black')
axes[1, 2].set_title('Phenytoin Residual Error', fontsize=14, fontweight='bold')
axes[1, 2].set_xlabel('Error (Actual - PPK)', fontsize=12)
axes[1, 2].set_ylabel('Frequency', fontsize=12)


sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df_filtered = df.copy()

# FEATURE EXTRACTION

feature_cols = [

    'anchor_age',
    'gender_encoded',
    'weight_kg',
    'creatinine',
    'albumin',
    'liver_health',
    'sepsis_severity',
    'infection_load',
    'heart_rate',
    'mean_bp',
    'wbc_count',
    'temperature_c',

    # PPK baseline estimates
    'ppk_vanco_estimate',
    'ppk_genta_estimate',
    'ppk_pheny_estimate'
]


X = df_filtered[feature_cols]

# Residual Errors = Target
y_vanco = df_filtered['error_vanco']
y_genta = df_filtered['error_genta']
y_pheny = df_filtered['error_pheny']

print("Feature Matrix Shape:", X.shape)


In [ ]:
# FEATURE CORRELATION HEATMAP

import matplotlib.pyplot as plt
import seaborn as sns


corr_matrix = X.corr()


plt.figure(figsize=(14,10))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)

plt.title(
    'Feature Correlation Heatmap',
    fontsize=16
)

plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# VANCOMYCIN LAB READINGS

vanco_table = df[['actual_vanco_conc']].copy()

vanco_table = vanco_table.rename(columns={
    'actual_vanco_conc': 'valuenum'
})

vanco_table['subject_id'] = np.arange(100000, 100000 + len(vanco_table))

vanco_table['charttime'] = pd.date_range(
    start='2025-01-01',
    periods=len(vanco_table),
    freq='H'
)

vanco_table['drug'] = 'Vancomycin'
vanco_table['valueuom'] = 'ug/mL'

vanco_table = vanco_table[
    ['subject_id', 'charttime', 'drug', 'valuenum', 'valueuom']
]

display(vanco_table.head())


In [ ]:
# GENTAMICIN LAB READINGS

genta_table = df[['actual_genta_conc']].copy()

genta_table = genta_table.rename(columns={
    'actual_genta_conc': 'valuenum'
})

genta_table['subject_id'] = np.arange(100000, 100000 + len(genta_table))

genta_table['charttime'] = pd.date_range(
    start='2025-01-01',
    periods=len(genta_table),
    freq='H'
)

genta_table['drug'] = 'Gentamicin'
genta_table['valueuom'] = 'ug/mL'

genta_table = genta_table[
    ['subject_id', 'charttime', 'drug', 'valuenum', 'valueuom']
]

display(genta_table.head())

In [ ]:
# PHENYTOIN LAB READINGS

pheny_table = df[['actual_pheny_conc']].copy()

pheny_table = pheny_table.rename(columns={
    'actual_pheny_conc': 'valuenum'
})

pheny_table['subject_id'] = np.arange(
    100000,
    100000 + len(pheny_table)
)

pheny_table['charttime'] = pd.date_range(
    start='2025-01-01',
    periods=len(pheny_table),
    freq='H'
)

pheny_table['drug'] = 'Phenytoin'
pheny_table['valueuom'] = 'ug/mL'

pheny_table = pheny_table[
    ['subject_id', 'charttime', 'drug', 'valuenum', 'valueuom']
]

display(pheny_table.head())


In [ ]:
# TRAIN TEST SPLIT — FEATURES

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
# VANCOMYCIN TARGET SPLIT

y_train_v = y_vanco.loc[X_train.index]

y_test_v = y_vanco.loc[X_test.index]

print("Vancomycin target split completed!")

In [ ]:
# GENTAMICIN TARGET SPLIT

y_train_g = y_genta.loc[X_train.index]

y_test_g = y_genta.loc[X_test.index]

print("Gentamicin target split completed!")

In [ ]:
# PHENYTOIN TARGET SPLIT

y_train_p = y_pheny.loc[X_train.index]

y_test_p = y_pheny.loc[X_test.index]

print("Phenytoin target split completed!")

In [ ]:
# INITIALIZING MACHINE LEARNING MODELS

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import xgboost as xgb

models = {

    'Linear Model': LinearRegression(),

    'Random Forest': RandomForestRegressor(
        n_estimators=50,
        random_state=42
    ),

    'SVM': SVR(),

    'XGBoost': xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=50,
        random_state=42
    )
}

print("Models initialized successfully!")

In [ ]:
# VANCOMYCIN HYBRID MODEL EVALUATION

results_vanco = []

true_v = df_filtered.loc[
    X_test.index,
    'actual_vanco_conc'
]

baseline_v = X_test['ppk_vanco_estimate']

print("VANCOMYCIN")

for model_name, model in models.items():

    # Train
    model.fit(X_train, y_train_v)

    # Predict residual error
    pred_error = model.predict(X_test)

    # Final hybrid prediction
    final_pred = baseline_v + pred_error

    # Metrics
    mae = mean_absolute_error(true_v, final_pred)

    rmse = np.sqrt(
        mean_squared_error(true_v, final_pred)
    )

    results_vanco.append({

        'Drug': 'Vancomycin',

        'Model': model_name,

        'Hybrid MAE': round(mae, 4),

        'Hybrid RMSE': round(rmse, 4)

    })

vanco_df = pd.DataFrame(results_vanco)

display(vanco_df.sort_values(by='Hybrid MAE'))

In [ ]:
# GENTAMICIN HYBRID MODEL EVALUATION

results_genta = []

true_g = df_filtered.loc[
    X_test.index,
    'actual_genta_conc'
]

baseline_g = X_test['ppk_genta_estimate']

print("GENTAMICIN")

for model_name, model in models.items():

    # Train
    model.fit(X_train, y_train_g)

    # Predict residual error
    pred_error = model.predict(X_test)

    # Final hybrid prediction
    final_pred = baseline_g + pred_error

    # Metrics
    mae = mean_absolute_error(true_g, final_pred)

    rmse = np.sqrt(
        mean_squared_error(true_g, final_pred)
    )

    results_genta.append({

        'Drug': 'Gentamicin',

        'Model': model_name,

        'Hybrid MAE': round(mae, 4),

        'Hybrid RMSE': round(rmse, 4)

    })

genta_df = pd.DataFrame(results_genta)

display(genta_df.sort_values(by='Hybrid MAE'))

In [ ]:
# PHENYTOIN HYBRID MODEL EVALUATION

results_pheny = []

true_p = df_filtered.loc[
    X_test.index,
    'actual_pheny_conc'
]

baseline_p = X_test['ppk_pheny_estimate']

print("PHENYTOIN")

for model_name, model in models.items():

    # Train
    model.fit(X_train, y_train_p)

    # Predict residual error
    pred_error = model.predict(X_test)

    # Final hybrid prediction
    final_pred = baseline_p + pred_error

    # Metrics
    mae = mean_absolute_error(true_p, final_pred)

    rmse = np.sqrt(
        mean_squared_error(true_p, final_pred)
    )

    results_pheny.append({

        'Drug': 'Phenytoin',

        'Model': model_name,

        'Hybrid MAE': round(mae, 4),

        'Hybrid RMSE': round(rmse, 4)

    })

pheny_df = pd.DataFrame(results_pheny)

display(pheny_df.sort_values(by='Hybrid MAE'))

In [ ]:
# MULTI-DRUG COMPARISON TABLE

comparison_df = pd.concat([

    vanco_df,
    genta_df,
    pheny_df

])


print(" MULTI-DRUG PERFORMANCE RESULTS ")


display(
    comparison_df.sort_values(
        by=['Drug', 'Hybrid MAE']
    )
)

In [ ]:
# MODEL COMPARISON CHART


import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")


plt.figure(figsize=(12,7), dpi=150)


palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']


ax = sns.barplot(
    data=comparison_df,
    x='Drug',
    y='Hybrid MAE',
    hue='Model',
    palette=palette
)


for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.2f',
        fontsize=9,
        padding=3
    )


plt.title(
    'Hybrid Model MAE Comparison',
    fontsize=20,
    fontweight='bold',
    pad=15
)


plt.xlabel(
    'Drug',
    fontsize=14,
    fontweight='bold'
)

plt.ylabel(
    'Mean Absolute Error (MAE)',
    fontsize=14,
    fontweight='bold'
)


plt.xticks(fontsize=11)
plt.yticks(fontsize=11)


plt.grid(
    axis='y',
    linestyle='--',
    alpha=0.25
)


plt.legend(
    title='Model',
    fontsize=10,
    title_fontsize=11,
    frameon=True
)


sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
# TRAINING RANDOM FOREST MODELS

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import pandas as pd
import numpy as np


#VANCOMYCIN 

rf_vanco = RandomForestRegressor(
    n_estimators=50,
    random_state=42
)

rf_vanco.fit(X_train, y_train_v)


#GENTAMICIN 

rf_genta = RandomForestRegressor(
    n_estimators=50,
    random_state=42
)

rf_genta.fit(X_train, y_train_g)


#PHENYTOIN

rf_pheny = RandomForestRegressor(
    n_estimators=50,
    random_state=42
)

rf_pheny.fit(X_train, y_train_p)

print("All Random Forest models trained successfully!")

In [ ]:
# VANCOMYCIN HYBRID RF RESULTS

pred_error_v = rf_vanco.predict(X_test)

baseline_v = X_test['ppk_vanco_estimate']

final_pred_v = baseline_v + pred_error_v

true_v = df_filtered.loc[
    X_test.index,
    'actual_vanco_conc'
]

print("VANCOMYCIN")

print(
    f"Baseline PPK MAE: "
    f"{mean_absolute_error(true_v, baseline_v):.2f}"
)

print(
    f"RF Hybrid ML MAE: "
    f"{mean_absolute_error(true_v, final_pred_v):.2f}"
)

print(
    f"Baseline PPK RMSE: "
    f"{np.sqrt(mean_squared_error(true_v, baseline_v)):.2f}"
)

print(
    f"RF Hybrid ML RMSE: "
    f"{np.sqrt(mean_squared_error(true_v, final_pred_v)):.2f}"
)

# Results Table
results_vanco = pd.DataFrame({

    'True_Vanco': true_v.round(2),

    'PPK_Vanco': baseline_v.round(2),

    'RF_Error_Vanco': pred_error_v.round(2),

    'Final_RF_Vanco': final_pred_v.round(2)

})

display(results_vanco.head())

In [ ]:
# GENTAMICIN HYBRID RF RESULTS

pred_error_g = rf_genta.predict(X_test)

baseline_g = X_test['ppk_genta_estimate']

final_pred_g = baseline_g + pred_error_g

true_g = df_filtered.loc[
    X_test.index,
    'actual_genta_conc'
]

print("GENTAMICIN")

print(
    f"Baseline PPK MAE: "
    f"{mean_absolute_error(true_g, baseline_g):.2f}"
)

print(
    f"RF Hybrid ML MAE: "
    f"{mean_absolute_error(true_g, final_pred_g):.2f}"
)

print(
    f"Baseline PPK RMSE: "
    f"{np.sqrt(mean_squared_error(true_g, baseline_g)):.2f}"
)

print(
    f"RF Hybrid ML RMSE: "
    f"{np.sqrt(mean_squared_error(true_g, final_pred_g)):.2f}"
)

# Results Table
results_genta = pd.DataFrame({

    'True_Genta': true_g.round(2),

    'PPK_Genta': baseline_g.round(2),

    'RF_Error_Genta': pred_error_g.round(2),

    'Final_RF_Genta': final_pred_g.round(2)

})

display(results_genta.head())

In [ ]:
# PHENYTOIN HYBRID RF RESULTS

pred_error_p = rf_pheny.predict(X_test)

baseline_p = X_test['ppk_pheny_estimate']

final_pred_p = baseline_p + pred_error_p

true_p = df_filtered.loc[
    X_test.index,
    'actual_pheny_conc'
]

print("PHENYTOIN")

print(
    f"Baseline PPK MAE: "
    f"{mean_absolute_error(true_p, baseline_p):.2f}"
)

print(
    f"RF Hybrid ML MAE: "
    f"{mean_absolute_error(true_p, final_pred_p):.2f}"
)

print(
    f"Baseline PPK RMSE: "
    f"{np.sqrt(mean_squared_error(true_p, baseline_p)):.2f}"
)

print(
    f"RF Hybrid ML RMSE: "
    f"{np.sqrt(mean_squared_error(true_p, final_pred_p)):.2f}"
)

# Results Table
results_pheny = pd.DataFrame({

    'True_Pheny': true_p.round(2),

    'PPK_Pheny': baseline_p.round(2),

    'RF_Error_Pheny': pred_error_p.round(2),

    'Final_RF_Pheny': final_pred_p.round(2)

})

display(results_pheny.head())

In [ ]:
# COMBINED ACTUAL vs PREDICTED SCATTER PLOT


import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))

# Vancomycin
plt.scatter(
    true_v,
    final_pred_v,
    alpha=0.7,
    label='Vancomycin'
)

# Gentamicin
plt.scatter(
    true_g,
    final_pred_g,
    alpha=0.7,
    label='Gentamicin'
)

# Phenytoin
plt.scatter(
    true_p,
    final_pred_p,
    alpha=0.7,
    label='Phenytoin'
)


all_true = list(true_v) + list(true_g) + list(true_p)

min_val = min(all_true)
max_val = max(all_true)

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle='--',
    linewidth=2,
    label='Perfect Prediction'
)

plt.title(
    'Actual vs Predicted Drug Concentrations',
    fontsize=18,
    fontweight='bold'
)

plt.xlabel(
    'Actual Concentration',
    fontsize=13
)

plt.ylabel(
    'Predicted Concentration',
    fontsize=13
)

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

baseline_vs_hybrid = pd.DataFrame({

    'Drug': [
        'Vancomycin',
        'Vancomycin',
        'Gentamicin',
        'Gentamicin',
        'Phenytoin',
        'Phenytoin'
    ],

    'Method': [
        'Baseline PPK',
        'Hybrid ML',
        'Baseline PPK',
        'Hybrid ML',
        'Baseline PPK',
        'Hybrid ML'
    ],

    'MAE': [
        mean_absolute_error(true_v, baseline_v),
        mean_absolute_error(true_v, final_pred_v),

        mean_absolute_error(true_g, baseline_g),
        mean_absolute_error(true_g, final_pred_g),

        mean_absolute_error(true_p, baseline_p),
        mean_absolute_error(true_p, final_pred_p)
    ]
})


plt.figure(figsize=(11,7))

ax = sns.barplot(
    data=baseline_vs_hybrid,
    x='Drug',
    y='MAE',
    hue='Method',
    palette=['#4C72B0', '#55A868']
)


for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.2f',
        padding=3,
        fontsize=10
    )


plt.title(
    'Baseline PPK vs Hybrid ML Performance',
    fontsize=20,
    fontweight='bold',
    pad=15
)

plt.xlabel(
    'Drug',
    fontsize=14,
    fontweight='bold'
)

plt.ylabel(
    'Mean Absolute Error (MAE)',
    fontsize=14,
    fontweight='bold'
)


plt.xticks(fontsize=12)
plt.yticks(fontsize=11)

plt.grid(
    axis='y',
    linestyle='--',
    alpha=0.4
)


plt.legend(
    title='Method',
    fontsize=11,
    title_fontsize=12,
    frameon=True
)


sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
# FEATURE IMPORTANCE GRAPH
# WITHOUT PPK + SEPSIS FEATURES


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Create dataframe
feature_importance_df = pd.DataFrame({

    'Feature': feature_cols,

    'Vancomycin': rf_vanco.feature_importances_,

    'Gentamicin': rf_genta.feature_importances_,

    'Phenytoin': rf_pheny.feature_importances_
})

# Average importance
feature_importance_df['Average Importance'] = (
    feature_importance_df[
        ['Vancomycin', 'Gentamicin', 'Phenytoin']
    ].mean(axis=1)
)

# REMOVE unwanted features
feature_importance_df = feature_importance_df[
    ~feature_importance_df['Feature'].isin([
        'ppk_vanco_estimate',
        'ppk_genta_estimate',
        'ppk_pheny_estimate',
        'sepsis_severity'
    ])
]

# Sort
feature_importance_df = feature_importance_df.sort_values(
    by='Average Importance',
    ascending=False
)

# Plot
plt.figure(figsize=(12,8))

# Different colors
colors = sns.color_palette(
    'viridis',
    len(feature_importance_df)
)

sns.barplot(
    data=feature_importance_df,
    x='Average Importance',
    y='Feature',
    palette=colors
)

# Title
plt.title(
    'Clinical Feature Importance Across Drug Models',
    fontsize=18,
    fontweight='bold'
)

# Labels
plt.xlabel(
    'Feature Importance Score',
    fontsize=13
)

plt.ylabel(
    'Clinical Features',
    fontsize=13
)

# Grid
plt.grid(
    axis='x',
    linestyle='--',
    alpha=0.5
)

plt.tight_layout()
plt.show()